# 2.1 Simulating an Income Distribution

In [66]:
import numpy as np
import matplotlib.pyplot as plt
import opgave2
import importlib

importlib.reload(opgave2)

education, ages, state, human_capital, income = opgave2.simulate_model()


# 2.2 Simulate the Income Distribution

In [ ]:
## Checking the Simulation
education_shares = np.bincount(education) / opgave2.N

print("Education shares:")
print(education_shares)

unemployment_by_age = np.mean(state == 1, axis=0)

print("Unemployment by age:")
print(unemployment_by_age)

print("Theoretical unemployment rate:")
print(opgave2.sigma / (opgave2.sigma + opgave2.lambda_))


## Income over the Life Cycle

In [ ]:

mean_income = np.mean(income, axis=0)

p10 = np.percentile(income, 10, axis=0)
p25 = np.percentile(income, 25, axis=0)
p50 = np.percentile(income, 50, axis=0)
p75 = np.percentile(income, 75, axis=0)
p90 = np.percentile(income, 90, axis=0)

# Plot mean income and selected percentiles

plt.figure(figsize=(10, 6))

plt.plot(ages, mean_income, label="Mean", linewidth=2)
plt.plot(ages, p10, label="10th percentile")
plt.plot(ages, p25, label="25th percentile")
plt.plot(ages, p50, label="Median")
plt.plot(ages, p75, label="75th percentile")
plt.plot(ages, p90, label="90th percentile")

plt.xlabel("Age")
plt.ylabel("Income")
plt.title("Income over the Life Cycle")
plt.legend()
plt.grid(True)
plt.xlim(18, 65)

plt.tight_layout()
plt.show()

## Income Distribution at Different Ages

In [ ]:

for age in [25, 35, 45, 60]:

    age_index = np.where(ages == age)[0][0]

    income_at_age = income[:, age_index]

    plt.figure(figsize=(8, 5))

    plt.hist(income_at_age, bins=30)

    plt.xlabel("Income")
    plt.ylabel("Number of individuals")
    plt.title(f"Income Distribution at Age {age}")

    plt.show()

## 2.3 Compute the Gini Coefficient

We compute the Gini coefficient of the simulated income distribution.
We first define our own Gini function and test it on distributions
where the theoretical answer is known.

In [ ]:
def gini(x):

    # Sort incomes from lowest to highest
    x = np.sort(x)

    # Number of observations
    n = len(x)

    # Calculate the Gini coefficient
    gini_value = (2* np.sum((np.arange(1, n + 1)) * x)/ (n * np.sum(x))- (n + 1) / n)

    return gini_value

### Testing the Gini function

We test the function using a uniform distribution on [0, 1].
The theoretical Gini coefficient is 1/3.

In [ ]:
# Uniform distribution

uniform_income = np.linspace(0, 1, 10000)

gini_uniform = gini(uniform_income)

print("Gini for uniform distribution:")
print(gini_uniform)

print("Theoretical Gini:")
print(1 / 3)

### Lognormal distribution

We also test the Gini function on a lognormal distribution.

In [ ]:
# Lognormal distribution

s = 0.5

rng_test = np.random.default_rng(123)

lognormal_income = rng_test.lognormal(0, s, 10000)

gini_lognormal = gini(lognormal_income)

print("Gini for lognormal distribution:")
print(gini_lognormal)

### Gini coefficient for the full simulated sample

We calculate the Gini coefficient for all simulated income observations pooled together.

In [ ]:
# Gini coefficient for all individuals and ages pooled

all_income = []

for i in range(len(income)):
    for j in range(len(ages)):
        all_income.append(income[i, j])

# Calculate the pooled Gini coefficient

gini_pooled = gini(all_income)

print("Gini coefficient for all individuals and ages pooled:")
print(gini_pooled)


# Gini coefficient by age

gini_by_age = []

for i in range(len(ages)):
    
    income_at_age = income[:, i]
    
    gini_at_age = gini(income_at_age)
    
    gini_by_age.append(gini_at_age)

print("Gini coefficient by age:")

for i in range(len(ages)):
    print(f"Age {ages[i]}: {gini_by_age[i]:.3f}")


# Plot Gini by age

plt.figure(figsize=(10, 6))

plt.plot(ages, gini_by_age)

plt.xlabel("Age")
plt.ylabel("Gini coefficient")
plt.title("Income Inequality by Age")

plt.xlim(18, 65)
plt.grid(True)

plt.tight_layout()
plt.show()

### Lorenz Curve

The Lorenz curve shows the cumulative share of total income received by the cumulative share of the population.

In [ ]:
# Sort incomes from lowest to highest

sorted_income = sorted(all_income)

# Total income

total_income = sum(sorted_income)

# Cumulative income

cumulative_income = []

current_income = 0

for income_value in sorted_income:
    
    current_income += income_value
    
    cumulative_income.append(current_income / total_income)


# Cumulative population share

cumulative_population = []

for i in range(len(sorted_income)):
    
    cumulative_population.append((i + 1) / len(sorted_income))

plt.figure(figsize=(8, 6))

# Lorenz curve

plt.plot(cumulative_population,  cumulative_income, label="Lorenz curve")

# Line of perfect equality

plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect equality")

plt.xlabel("Cumulative share of population")
plt.ylabel("Cumulative share of income")
plt.title("Lorenz Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Pooled versus within-age inequality

The pooled Gini coefficient measures inequality when individuals of all ages are considered together. It therefore captures both differences in income within age groups and differences in income across ages.

The Gini coefficients calculated separately by age measure inequality only among individuals of the same age. These coefficients show how income inequality develops over the life cycle.

The pooled Gini coefficient is higher than the inequality within most individual age groups. This indicates that differences across ages contribute to overall income inequality, in addition to differences between individuals of the same age.

In [ ]:
# Compare pooled inequality with inequality within age groups

print("Pooled Gini coefficient:")
print(f"{gini_pooled:.3f}")

print("\nGini coefficient for selected ages:")

for age in [25, 35, 45, 55, 65]:

    age_index = np.where(ages == age)[0][0]

    print(f"Age {age}: {gini_by_age[age_index]:.3f}")

## 2.4 What drives inequality?

We investigate which mechanisms in the model are important for generating income inequality.
We start by removing educational differences while keeping the other mechanisms unchanged.

In [ ]:
# Reload the updated model

import importlib
importlib.reload(opgave2)


# Run the model without educational differences

education_no, ages_no, state_no, human_capital_no, income_no = opgave2.simulate_no_education()

In [ ]:
# Collect all income observations

all_income_no = []

for i in range(len(income_no)):
    for j in range(len(ages_no)):
        all_income_no.append(income_no[i, j])


# Calculate pooled Gini

gini_no_education = gini(all_income_no)

print("Pooled Gini without education differences:")
print(f"{gini_no_education:.3f}")

# Gini at age 45

age = 45

age_index = np.where(ages_no == age)[0][0]

gini_no_education_45 = gini(income_no[:, age_index])

print("Gini at age 45 without education differences:")
print(f"{gini_no_education_45:.3f}")

In [ ]:
education_shock, ages_shock, state_shock, human_capital_shock, income_shock = opgave2.simulate_no_shocks()

# Collect all income observations

all_income_shock = []

for i in range(len(income_shock)):
    for j in range(len(ages_shock)):
        all_income_shock.append(income_shock[i, j])


# Calculate pooled Gini

gini_no_shocks = gini(all_income_shock)

print("Pooled Gini without human capital shocks:")
print(f"{gini_no_shocks:.3f}")

# Gini at age 45

age = 45

age_index = np.where(ages_shock == age)[0][0]

gini_no_shocks_45 = gini(income_shock[:, age_index])

print("Gini at age 45 without human capital shocks:")
print(f"{gini_no_shocks_45:.3f}")

In [ ]:
importlib.reload(opgave2)
education_dep, ages_dep, state_dep, human_capital_dep, income_dep = opgave2.simulate_no_depreciation()
all_income_dep = []

for i in range(len(income_dep)):
    for j in range(len(ages_dep)):
        all_income_dep.append(income_dep[i, j])

gini_no_depreciation = gini(all_income_dep)

print("Pooled Gini without depreciation:")
print(f"{gini_no_depreciation:.3f}")

age = 45

age_index = np.where(ages_dep == age)[0][0]

gini_no_depreciation_45 = gini(income_dep[:, age_index])

print("Gini at age 45 without depreciation:")
print(f"{gini_no_depreciation_45:.3f}")

In [ ]:
importlib.reload(opgave2)

education_unemp, ages_unemp, state_unemp, human_capital_unemp, income_unemp = opgave2.simulate_no_unemployment()

# Collect all income observations

all_income_unemp = []

for i in range(len(income_unemp)):
    for j in range(len(ages_unemp)):
        all_income_unemp.append(income_unemp[i, j])


# Calculate pooled Gini

gini_no_unemployment = gini(all_income_unemp)

print("Pooled Gini without unemployment:")
print(f"{gini_no_unemployment:.3f}")

# Gini at age 45

age = 45

age_index = np.where(ages_unemp == age)[0][0]

gini_no_unemployment_45 = gini(income_unemp[:, age_index])

print("Gini at age 45 without unemployment:")
print(f"{gini_no_unemployment_45:.3f}")

### Results

The baseline pooled Gini coefficient is 0.377. Removing educational differences reduces the pooled Gini to 0.298, showing that education is an important source of income inequality in the model.

Removing human capital shocks has the largest effect on the pooled Gini, which falls to 0.280. The Gini coefficient at age 45 also falls substantially to 0.215. This indicates that human capital shocks are an important source of both overall and within-age inequality.

Removing depreciation while unemployed increases the pooled Gini slightly from 0.377 to 0.382. Similarly, removing unemployment increases the pooled Gini to 0.387. Thus, in this model, depreciation and unemployment have a small equalising effect on the income distribution.

Overall, the results suggest that human capital shocks and educational differences are important drivers of inequality, while unemployment and depreciation reduce inequality slightly in the model.

## 2.5 Extension: Health Risk

We extend the model by introducing health risk as an additional source of uncertainty. The probability of becoming sick is low at young ages and increases with age. Sick individuals can recover in the following period, with the probability of recovery decreasing with age.

For employed individuals, being sick reduces income to 70% of their normal income.

In [ ]:
importlib.reload(opgave2)
education_health, ages_health, state_health, human_capital_health, income_health, healthy = opgave2.simulate_health_risk()
all_income_health = []

for i in range(len(income_health)):
    for j in range(len(ages_health)):
        all_income_health.append(income_health[i, j])

gini_health = gini(all_income_health)

print("Pooled Gini with health risk:")
print(f"{gini_health:.3f}")

age = 45

age_index = np.where(ages_health == age)[0][0]

gini_health_45 = gini(income_health[:, age_index])

print("Gini at age 45 with health risk:")
print(f"{gini_health_45:.3f}")

The pooled Gini coefficient is 0.376 with health risk, compared with 0.377 in the baseline model. Thus, the additional health risk has only a very small effect on overall income inequality in our simulation.

At age 45, the Gini coefficient is 0.339. This shows that health risk introduces some additional variation in income within the age group, although its overall effect on inequality is relatively limited.